# Table 7 Torus Covering Fraction Versus Redshift

This notebook reads `table_7.csv`, joins Table 7 redshifts to JAXSEDFit fit results containing the AGN torus `fcov` parameter, and plots `fcov` as a function of redshift. Table 7 does not contain `fcov` directly, so the optional fit section below can generate a resumable `fcov` result table from Table 7 targets.

In [1]:
from pathlib import Path
import csv
import sys

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table


def find_repo_root():
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (path / "src" / "jaxsedfit").is_dir() and (path / "table_7.csv").is_file():
            return path
        nested = path / "jaxsedfit"
        if (nested / "src" / "jaxsedfit").is_dir() and (nested / "table_7.csv").is_file():
            return nested
    raise RuntimeError("Could not find the jaxsedfit repository root with table_7.csv")


ROOT = find_repo_root()
TABLE7_PATH = ROOT / "table_7.csv"
OUTPUT_DIR = ROOT / "notebook_outputs" / "17_table7_fcov_redshift"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FCOV_RESULTS_PATH = OUTPUT_DIR / "table7_fcov_fit_results.ecsv"
FCOV_FAILURES_PATH = OUTPUT_DIR / "table7_fcov_fit_failures.ecsv"
FCOV_FIGURE_PATH = OUTPUT_DIR / "table7_fcov_vs_redshift.png"

print(f"Reading: {TABLE7_PATH}")
print(f"Results path: {FCOV_RESULTS_PATH}")

Reading: /home/nicho/GRAHSP/jaxsedfit/table_7.csv
Results path: /home/nicho/GRAHSP/jaxsedfit/notebook_outputs/17_table7_fcov_redshift/table7_fcov_fit_results.ecsv


## Load Table 7

In [2]:
def as_float(value, default=np.nan):
    try:
        text = str(value).strip()
        if text == "" or text.lower() in {"nan", "none", "--"}:
            return default
        return float(text)
    except Exception:
        return default


with TABLE7_PATH.open(newline="") as handle:
    table7_rows = list(csv.DictReader(handle))

table7_by_name = {row["SDSS Name"]: row for row in table7_rows}
sdss_name = np.array([row["SDSS Name"] for row in table7_rows], dtype=str)
redshift = np.array([as_float(row["z"]) for row in table7_rows], dtype=float)
redshift_err = np.array([as_float(row["z_err"]) for row in table7_rows], dtype=float)
m_2500 = np.array([as_float(row["m_2500"]) for row in table7_rows], dtype=float)

valid_table7 = np.isfinite(redshift)
print(f"Loaded {len(table7_rows):,} rows from Table 7")
print(f"Rows with finite redshift: {valid_table7.sum():,}")

Loaded 6,007 rows from Table 7
Rows with finite redshift: 6,007


## Plot Existing `fcov` Results

In [3]:
def finite_column(table, name, default=np.nan):
    if name in table.colnames:
        return np.asarray(table[name], dtype=float)
    return np.full(len(table), default, dtype=float)


def table7_lookup_float(source_ids, column):
    return np.array([as_float(table7_by_name.get(str(source_id), {}).get(column, np.nan)) for source_id in source_ids], dtype=float)


def plot_fcov_vs_redshift(results, output_path=FCOV_FIGURE_PATH, show=True):
    source_ids = np.asarray(results["source_id"], dtype=str)
    z = finite_column(results, "redshift")
    z_err = finite_column(results, "redshift_err")
    if not np.isfinite(z_err).any():
        z_err = table7_lookup_float(source_ids, "z_err")
    z_err = np.where(np.isfinite(z_err) & (z_err > 0), z_err, 0.0)

    fcov = finite_column(results, "fcov_median")
    fcov_lo = finite_column(results, "fcov_p16")
    fcov_hi = finite_column(results, "fcov_p84")
    if not np.isfinite(fcov).any() and "fcov" in results.colnames:
        fcov = finite_column(results, "fcov")
    yerr = None
    if np.isfinite(fcov_lo).any() and np.isfinite(fcov_hi).any():
        yerr = np.vstack([
            np.maximum(fcov - fcov_lo, 0.0),
            np.maximum(fcov_hi - fcov, 0.0),
        ])

    color = finite_column(results, "hot_fcov_median")
    color_label = r"hot dust $f_{cov}$"
    if not np.isfinite(color).any():
        color = table7_lookup_float(source_ids, "m_2500")
        color_label = r"$m_{2500}$"

    finite = np.isfinite(z) & np.isfinite(fcov)
    if not finite.any():
        raise ValueError("No finite fcov/redshift pairs were found in the result table.")

    fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
    ax.errorbar(
        z[finite],
        fcov[finite],
        xerr=z_err[finite],
        yerr=yerr[:, finite] if yerr is not None else None,
        fmt="none",
        ecolor="0.72",
        elinewidth=0.7,
        alpha=0.55,
        zorder=1,
    )
    scatter = ax.scatter(
        z[finite],
        fcov[finite],
        c=color[finite] if np.isfinite(color[finite]).any() else None,
        s=24,
        cmap="viridis",
        alpha=0.78,
        linewidths=0,
        zorder=2,
    )

    if finite.sum() >= 5:
        bins = np.linspace(np.nanmin(z[finite]), np.nanmax(z[finite]), 13)
        centers = 0.5 * (bins[:-1] + bins[1:])
        medians = []
        counts = []
        for left, right in zip(bins[:-1], bins[1:]):
            in_bin = (z >= left) & (z < right) & finite
            counts.append(in_bin.sum())
            medians.append(np.nanmedian(fcov[in_bin]) if in_bin.any() else np.nan)
        medians = np.asarray(medians)
        counts = np.asarray(counts)
        good_bins = np.isfinite(medians) & (counts > 0)
        ax.plot(
            centers[good_bins],
            medians[good_bins],
            color="black",
            marker="o",
            linewidth=1.8,
            markersize=4,
            label="Median in redshift bins",
            zorder=3,
        )
        ax.legend(frameon=False, loc="best")

    ax.set_xlabel("Redshift, z")
    ax.set_ylabel(r"AGN torus covering fraction, $f_{cov}$")
    ax.set_title("Table 7 JAXSEDFit Torus Covering Fraction Versus Redshift")
    ax.set_ylim(0.0, max(1.0, np.nanmax(fcov[finite]) * 1.08))
    ax.grid(alpha=0.22)
    if np.isfinite(color[finite]).any():
        cbar = fig.colorbar(scatter, ax=ax, pad=0.015)
        cbar.set_label(color_label)

    fig.savefig(output_path, dpi=220)
    print(f"Plotted {finite.sum():,} fitted Table 7 sources")
    print(f"Saved figure to {output_path}")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


if FCOV_RESULTS_PATH.exists():
    fcov_results = Table.read(FCOV_RESULTS_PATH, format="ascii.ecsv")
    print(f"Loaded {len(fcov_results):,} fcov fit rows from {FCOV_RESULTS_PATH}")
    plot_fcov_vs_redshift(fcov_results)
else:
    print(f"No fcov result table found yet at {FCOV_RESULTS_PATH}")
    print("Run the optional fitting section below to create one.")

No fcov result table found yet at /home/nicho/GRAHSP/jaxsedfit/notebook_outputs/17_table7_fcov_redshift/table7_fcov_fit_results.ecsv
Run the optional fitting section below to create one.


## Optional Resumable Fits

Set `RUN_TABLE7_FCOV_FITS = True` to query broadband photometry for Table 7 sources, fit each source with JAXSEDFit, save `fcov` and `hot_fcov`, and then re-run the plot above.

In [4]:
RUN_TABLE7_FCOV_FITS = True

FCOV_MAX_OBJECTS = 25
FCOV_BATCH_SIZE = 5
FCOV_FIT_STEPS = 800
FCOV_FIT_LEARNING_RATE = 5e-3
FCOV_MIN_BANDS_TO_FIT = 5
FCOV_MAX_MAG_ERR = 1.0
FCOV_RANDOM_SEED = 20260601

FILTER_SPECLITE_NAME = {
    "FUV_galex": "galex-fuv",
    "NUV_galex": "galex-nuv",
    "u_sdss": "sdss2010-u",
    "g_sdss": "sdss2010-g",
    "r_sdss": "sdss2010-r",
    "i_sdss": "sdss2010-i",
    "z_sdss": "sdss2010-z",
    "W1": "wise2010-W1",
    "W2": "wise2010-W2",
    "W3": "wise2010-W3",
    "W4": "wise2010-W4",
}
FILTER_ORDER = {name: i for i, name in enumerate(FILTER_SPECLITE_NAME)}

In [5]:
if RUN_TABLE7_FCOV_FITS:
    import gc
    import astropy.units as u
    from astropy.coordinates import SkyCoord
    try:
        import jax
    except Exception:
        jax = None

    for path in (ROOT / "src", ROOT.parent / "bandwagon" / "src"):
        if path.is_dir() and str(path) not in sys.path:
            sys.path.insert(0, str(path))

    from bandwagon import matches_to_photometry, xmatch_catalogs
    from jaxsedfit.config import (
        AGNConfig,
        FilterSet,
        FitConfig,
        GalaxyConfig,
        InferenceConfig,
        LikelihoodConfig,
        Observation,
        PhotometryData,
    )
    from jaxsedfit.core import JAXSEDFit

    dsps_ssp_fn = ROOT / "tempdata.h5"
    assert dsps_ssp_fn.is_file(), f"DSPS SSP file not found: {dsps_ssp_fn}"

    def read_rows(path):
        if not path.exists():
            return []
        return [dict(row) for row in Table.read(path, format="ascii.ecsv")]

    def write_rows(rows_to_write, path):
        if rows_to_write:
            tmp_path = path.with_suffix(path.suffix + ".tmp")
            Table(rows=rows_to_write).write(tmp_path, format="ascii.ecsv", overwrite=True)
            tmp_path.replace(path)

    def save_progress():
        write_rows(fcov_success_rows, FCOV_RESULTS_PATH)
        write_rows(fcov_failure_rows, FCOV_FAILURES_PATH)

    def cleanup_memory():
        gc.collect()
        if jax is not None:
            try:
                jax.clear_caches()
            except Exception:
                pass

    def scalar_float(value, default=np.nan):
        try:
            arr = np.asarray(value, dtype=float).reshape(-1)
            return float(arr[0]) if arr.size else default
        except Exception:
            return as_float(value, default=default)

    def rows_for_source(photometry_table, source_id):
        mask = np.asarray(photometry_table["source_id"], dtype=str) == str(source_id)
        source_rows = photometry_table[mask]
        order = np.argsort([FILTER_ORDER.get(str(name), 999) for name in source_rows["filter_name"]])
        return source_rows[order]

    def build_fit_config(source_id, source_rows):
        table_row = table7_by_name[str(source_id)]
        fluxes = np.asarray(source_rows["flux_mjy"], dtype=float)
        errors = np.asarray(source_rows["flux_err_mjy"], dtype=float)
        errors = np.maximum(errors, 0.03 * fluxes)
        psf = [as_float(value) for value in source_rows["psf_fwhm_arcsec"]] if "psf_fwhm_arcsec" in source_rows.colnames else [np.nan] * len(source_rows)

        return FitConfig(
            observation=Observation(
                object_id=str(source_id),
                redshift=as_float(table_row["z"]),
                fit_redshift=False,
                ra=as_float(table_row["RA"]),
                dec=as_float(table_row["Dec"]),
            ),
            photometry=PhotometryData(
                filter_names=[str(name) for name in source_rows["filter_name"]],
                fluxes=fluxes.tolist(),
                errors=errors.tolist(),
                is_upper_limit=[False] * len(source_rows),
                psf_fwhm_arcsec=psf,
            ),
            filters=FilterSet(
                speclite_names={str(row["filter_name"]): str(row["speclite_name"]) for row in source_rows},
                use_grahsp_database=False,
            ),
            galaxy=GalaxyConfig(dsps_ssp_fn=str(dsps_ssp_fn), n_wave=768),
            agn=AGNConfig(agn_type=1),
            likelihood=LikelihoodConfig(
                systematics_width=0.08,
                variability_uncertainty=True,
                use_host_capture_model=True,
            ),
            inference=InferenceConfig(
                map_steps=FCOV_FIT_STEPS,
                learning_rate=FCOV_FIT_LEARNING_RATE,
                seed=FCOV_RANDOM_SEED,
            ),
            prior_config={
                "log_stellar_mass": {"loc": 10.5, "scale": 1.25},
                "fracAGN_5100": {"loc": 0.75, "scale": 0.2},
                "ebv_gal": {"scale": 0.2},
                "ebv_agn": {"scale": 0.2},
            },
        )


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
/home/nicho/miniconda3/envs/astro/lib/python3.11/site-packages/dustmaps/config.py:74: ConfigWarning: Configuration file not found:

    /home/nicho/.dustmapsrc

To create a new configuration file in the default location, run the following python code:

    from dustmaps.config import config
    config.reset()

Note that this will delete your configuration! For example, if you have specified a data directory, then dustmaps will forget about its location.
  warn(('Configuration file not found:\n\n'


In [6]:
if RUN_TABLE7_FCOV_FITS:
    fcov_success_rows = read_rows(FCOV_RESULTS_PATH)
    fcov_failure_rows = read_rows(FCOV_FAILURES_PATH)
    completed_ids = {str(row["source_id"]) for row in fcov_success_rows}
    failed_ids = {str(row["source_id"]) for row in fcov_failure_rows}
    attempted_ids = completed_ids | failed_ids

    target_indices = np.flatnonzero(valid_table7)
    if FCOV_MAX_OBJECTS is not None:
        target_indices = target_indices[: int(FCOV_MAX_OBJECTS)]
    target_rows = [table7_rows[int(idx)] for idx in target_indices]
    pending_rows = [row for row in target_rows if str(row["SDSS Name"]) not in attempted_ids]

    print(f"Target Table 7 objects: {len(target_rows):,}")
    print(f"Already successful: {len(completed_ids):,}")
    print(f"Already failed/skipped: {len(failed_ids):,}")
    print(f"Pending: {len(pending_rows):,}")

Target Table 7 objects: 25
Already successful: 0
Already failed/skipped: 0
Pending: 25


In [7]:
if RUN_TABLE7_FCOV_FITS:
    for batch_start in range(0, len(pending_rows), FCOV_BATCH_SIZE):
        batch_rows = pending_rows[batch_start : batch_start + FCOV_BATCH_SIZE]
        batch_ids = np.array([str(row["SDSS Name"]) for row in batch_rows], dtype=str)
        batch_ra = np.array([as_float(row["RA"]) for row in batch_rows], dtype=float)
        batch_dec = np.array([as_float(row["Dec"]) for row in batch_rows], dtype=float)
        finite_coords = np.isfinite(batch_ra) & np.isfinite(batch_dec)

        if not finite_coords.any():
            for source_id in batch_ids:
                fcov_failure_rows.append({"source_id": str(source_id), "reason": "missing RA/Dec"})
            save_progress()
            cleanup_memory()
            continue

        query_ids = batch_ids[finite_coords]
        query_coords = SkyCoord(ra=batch_ra[finite_coords] * u.deg, dec=batch_dec[finite_coords] * u.deg, frame="icrs")
        print(f"Batch {batch_start // FCOV_BATCH_SIZE + 1}: querying and fitting {len(query_ids)} objects")

        try:
            batch_matches = xmatch_catalogs(query_coords, source_id=query_ids)
            batch_photometry = matches_to_photometry(batch_matches, max_mag_err=FCOV_MAX_MAG_ERR)
        except Exception as exc:
            for source_id in query_ids:
                fcov_failure_rows.append({"source_id": str(source_id), "reason": f"xmatch failed: {exc}"})
            save_progress()
            cleanup_memory()
            continue

        for source_id in query_ids:
            try:
                source_rows = rows_for_source(batch_photometry, source_id) if len(batch_photometry) else batch_photometry
                if len(source_rows) < FCOV_MIN_BANDS_TO_FIT:
                    fcov_failure_rows.append({"source_id": str(source_id), "reason": f"only {len(source_rows)} usable bands"})
                else:
                    cfg = build_fit_config(source_id, source_rows)
                    fitter = JAXSEDFit(cfg)
                    map_result = fitter.fit_map(
                        steps=FCOV_FIT_STEPS,
                        learning_rate=FCOV_FIT_LEARNING_RATE,
                        progress_bar=False,
                    )
                    med = fitter.map_result["median"]
                    table_row = table7_by_name[str(source_id)]
                    losses = np.asarray(map_result.get("losses", [np.nan]), dtype=float)
                    fcov_success_rows.append({
                        "source_id": str(source_id),
                        "redshift": as_float(table_row["z"]),
                        "redshift_err": as_float(table_row["z_err"]),
                        "m_2500": as_float(table_row["m_2500"]),
                        "n_bands": len(source_rows),
                        "fcov_median": scalar_float(med.get("fcov", np.nan)),
                        "hot_fcov_median": scalar_float(med.get("hot_fcov", np.nan)),
                        "log_agn_amp_map": scalar_float(med.get("log_agn_amp", np.nan)),
                        "final_loss": float(losses[-1]) if losses.size else np.nan,
                    })
                save_progress()
                print(f"Saved progress after {source_id}: {len(fcov_success_rows):,} successes, {len(fcov_failure_rows):,} failures/skips")
            except Exception as exc:
                fcov_failure_rows.append({"source_id": str(source_id), "reason": f"fit failed: {exc}"})
                save_progress()
            finally:
                cleanup_memory()

        cleanup_memory()

Batch 1: querying and fitting 5 objects


KeyboardInterrupt: 

In [ ]:
if FCOV_RESULTS_PATH.exists():
    fcov_results = Table.read(FCOV_RESULTS_PATH, format="ascii.ecsv")
    plot_fcov_vs_redshift(fcov_results)
else:
    print(f"No fcov result table found yet at {FCOV_RESULTS_PATH}")

No fcov result table found yet at /home/nicho/GRAHSP/jaxsedfit/notebook_outputs/17_table7_fcov_redshift/table7_fcov_fit_results.ecsv
